# 04 -- Session and news analysis

Paired script: `analysis/join_news_events.py` for the NEWS half. Independently recomputes
`NEWS_BLACKOUT` status (per `NewsManager.mqh`/section 10) for every journal decision,
directly useful given every real journal record's `news_state` is currently always empty
(the live EA never sets it -- see `analysis/schema.py`'s docstring).

**Fixed, 2026-07-21 Codex review finding:** this notebook previously performed only the news
join and no SESSION analysis at all, despite the name. It now also groups synthetic decisions
by trading session (derived from the decision's own UTC hour, matching a common London/
New York/Asia session convention) and reports win rate by session -- the second half of what
this notebook's name promises.

**Uses clearly-labelled SYNTHETIC journal/news/trade fixtures.** Real-data run: PENDING.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_news_events import run as run_news_join
from analysis.metrics import win_rate

## News-blackout join

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_news_demo_"))

decision = {
    "signal_id": "sig-1", "timestamp_utc": "2026-07-21T14:05:30Z", "symbol": "XAUUSD",
    "market_family": "METAL", "intraday_mode": "SCALP", "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5, "direction": "BUY", "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback", "candlestick_pattern": None, "chart_pattern": None,
    "score": 68.0, "score_breakdown": {}, "entry": 2350.55, "stop": 2345.10,
    "targets": [2361.45], "risk_percent": 0.3, "news_state": "", "session_state": "",
    "reasons_passed": [], "reasons_rejected": [], "ea_version": "1.01", "git_commit": "abc",
}
(tmp_dir / "decisions_20260721.jsonl").write_text(json.dumps(decision) + "\n", encoding="utf-8")

pd.DataFrame([{
    "event_id": "e-nfp", "event_name": "NFP", "currency": "USD", "importance": 2,
    "scheduled_utc": "2026-07-21T14:10:00Z",  # 4m30s after the decision -- inside the window
}]).to_csv(tmp_dir / "news.csv", index=False)

news_result = run_news_join(tmp_dir, tmp_dir / "news.csv", currency="USD", before_minutes=15,
                             after_minutes=15, min_importance=2, repo_path=PROJECT_ROOT.parents[1])

print(f"n_decisions      = {news_result.n_decisions}")
print(f"n_in_blackout    = {news_result.n_in_blackout}")

assert news_result.n_in_blackout == 1
assert news_result.joined.iloc[0]["triggering_event_id"] == "e-nfp"

## Session analysis (synthetic)

No dedicated "session" script exists yet in this project's required-scripts list, and
`session_state` is (like `news_state`) always empty in the live EA's real output today. This
cell demonstrates the shape of a session breakdown -- win rate grouped by a
London/New-York/Asia session bucket derived from each decision's own UTC hour -- using
synthetic decisions, since no real journal spans enough real trading hours yet to do this for
real.

In [ ]:
def session_for_hour(hour_utc: int) -> str:
    # A commonly-used, deliberately simple UTC-hour session convention --
    # not sourced from any MQL5 SessionManager.mqh formula (that module
    # reads the broker's own session times, which this synthetic demo
    # cannot access), stated as an explicit simplification.
    if 0 <= hour_utc < 7:
        return "asia"
    if 7 <= hour_utc < 12:
        return "london"
    return "new_york"

synthetic_decisions = pd.DataFrame({
    "signal_id": [f"s{i}" for i in range(9)],
    "hour_utc": [2, 3, 4, 8, 9, 10, 14, 15, 16],
    "profit": [10.0, -5.0, 10.0, 10.0, 10.0, -5.0, -5.0, -5.0, 10.0],
})
synthetic_decisions["session"] = synthetic_decisions["hour_utc"].apply(session_for_hour)

rows = []
for session_label, group in synthetic_decisions.groupby("session"):
    wr = win_rate((group["profit"] > 0).tolist())
    rows.append({"session": session_label, "n": wr.n, "win_rate": wr.win_rate,
                 "win_rate_ci_lower": wr.ci_lower, "win_rate_ci_upper": wr.ci_upper})

performance_by_session = pd.DataFrame(rows)
print(performance_by_session)

assert len(performance_by_session) == 3
asia_row = performance_by_session[performance_by_session["session"] == "asia"].iloc[0]
assert abs(asia_row["win_rate"] - (2.0 / 3.0)) < 1e-9  # 2 wins of 3

## Real-data run: PENDING

Requires a real journal (batched runtime verification, TASK-025+), a real news-event export,
and enough real decisions spanning multiple sessions -- none exist yet. A real session
breakdown should also use `SessionManager.mqh`'s own broker-session-time logic (ported to
Python, not yet done) rather than this notebook's simplified fixed UTC-hour buckets.